# [센서 실습 7] 적외선(IR) 불꽃 센서 – KY-026 불꽃 감지 모듈

불꽃에서 나오는 적외선을 감지하는 **KY-026** 모듈의 아날로그 출력을 ADC(MCP3008)로 읽어 불꽃 세기를 구하고, 연속 감지 조건으로 화재 경보를 판정합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
불꽃은 **760~1100nm 근적외선**을 강하게 내보냅니다. KY-026은 이 파장에 민감한 **IR 수광 다이오드**를 사용합니다.
- 적외선이 강할수록 출력 전압이 **낮아짐** (A0: 아날로그 출력)
- 비교기가 기준과 비교한 디지털 출력도 있음 (D0)
- 라즈베리파이에는 **아날로그 입력이 없으므로** ADC 칩(MCP3008)이 전압을 숫자(0~1023)로 바꿔 줌
- 햇빛·백열등에도 적외선이 있어 **오경보**가 날 수 있음 → 연속 감지 조건이 필요

### 부품 사양
| 항목 | 내용 |
|---|---|
| 감지 파장 | 760 ~ 1100 nm |
| 감지 거리 | 약 80cm (라이터 불꽃 기준) |
| 감지 각도 | 약 60° |
| 출력 | A0 아날로그 + D0 디지털 |
| 전원 | 3.3 ~ 5V |

## 2. 라즈베리파이 연결 방법
**준비물**: KY-026 모듈, MCP3008 ADC, 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MCP3008 VDD, VREF (16, 15번) | 3.3V (1번 핀) | ADC 전원과 기준 전압 |
| MCP3008 AGND, DGND (14, 9번) | GND (6번 핀) |  |
| MCP3008 CLK (13번) | GPIO11 / SCLK (23번 핀) | SPI 클록 |
| MCP3008 DOUT (12번) | GPIO9 / MISO (21번 핀) | SPI 데이터 (ADC → Pi) |
| MCP3008 DIN (11번) | GPIO10 / MOSI (19번 핀) | SPI 데이터 (Pi → ADC) |
| MCP3008 CS (10번) | GPIO8 / CE0 (24번 핀) | 칩 선택 |
| KY-026 VCC / GND | 3.3V / GND |  |
| KY-026 A0 | MCP3008 CH0 (1번) | 아날로그 출력 |
| KY-026 D0 | GPIO27 (13번 핀) | 디지털 출력 (선택) |

> 💡 MCP3008은 **10비트 ADC**입니다. 0~3.3V를 0~1023으로 바꾸며, `gpiozero`는 이를 0.0~1.0으로 정규화한 `value`를 줍니다.

**인터페이스 활성화**
- **SPI**: `sudo raspi-config` → Interface Options → **SPI** → Enable
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
ADC 값(0.0~1.0)을 전압과 **불꽃 세기(%)** 로 바꿉니다. 불꽃이 셀수록 전압이 낮아지므로 뒤집어서 표현합니다.

In [ ]:
VREF = 3.3

def adc_to_voltage(adc_value):
    """gpiozero MCP3008.value (0.0~1.0) → 전압(V)"""
    return adc_value * VREF

def flame_intensity(adc_value):
    """적외선이 강할수록 전압이 낮아지므로 0~100%로 뒤집어 표현"""
    return round((1 - adc_value) * 100, 1)

print(f"ADC 0.93 → {adc_to_voltage(0.93):.2f}V, 불꽃 세기 {flame_intensity(0.93)}%")

### 4-2. 실제 센서 클래스
`gpiozero.MCP3008`로 아날로그 값을, `DigitalInputDevice`로 디지털 출력을 함께 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealFlameIR:
    def __init__(self, channel=0, do_pin=27):
        from gpiozero import MCP3008, DigitalInputDevice
        self.adc = MCP3008(channel=channel)
        self.do = DigitalInputDevice(do_pin, pull_up=None, active_state=True)

    def read(self):
        v = self.adc.value
        return {"adc": round(v, 4), "ir_pct": flame_intensity(v), "d0": int(self.do.value)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 평상시 | 불꽃 세기 3~8% | 실내 조명·주변 적외선 |
| 햇빛 반사 | 20~25초에 잠깐 40% | 창문 햇빛이 비쳐 생기는 오경보 요인 |
| 불꽃 | 60~80초에 라이터 불꽃, 70~90% | 30cm 거리 라이터 |
| 깜빡임 | 불꽃 세기 ±8% 흔들림 | 불꽃은 계속 흔들림 |
| D0 출력 | 세기 50% 이상이면 0 | 모듈 비교기(가변저항) 설정값 |

가상 센서는 불꽃 세기로부터 **ADC 값(0.0~1.0)** 을 만들어 실제와 같은 변환식을 통과시킵니다.

In [ ]:
class VirtualFlameIR:
    def __init__(self, dt=0.5, seed=7):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def read(self):
        ir = 5 + self.rng.normal(0, 1.5)                          # 평상시 주변 적외선(%)
        if 20 <= self.t < 25:
            ir = 40 + self.rng.normal(0, 3)                       # 햇빛 반사 (짧은 오경보 요인)
        if 60 <= self.t < 80:
            ir = 80 + self.rng.normal(0, 8)                       # 라이터 불꽃
        ir = float(np.clip(ir, 0, 100))
        adc = round(1 - ir / 100, 4)                              # 세기가 클수록 전압이 낮음
        self.t += self.dt
        return {"adc": adc, "ir_pct": flame_intensity(adc), "d0": 0 if ir >= 50 else 1}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualFlameIR(dt=0.5)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
0.5초 간격으로 2분 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 240               # 측정 횟수
INTERVAL = 0.5               # 측정 간격(초)

sensor = RealFlameIR() if USE_REAL_SENSOR else VirtualFlameIR(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
불꽃 세기 **50% 이상이 3회 연속(1.5초)** 이어질 때만 화재 경보를 울려, 순간적인 햇빛 반사 같은 오경보를 줄입니다.

In [ ]:
TH, CONSEC = 50, 3
df["over"] = (df["ir_pct"] >= TH).astype(int)
df["alarm"] = (df["over"].rolling(CONSEC).sum() == CONSEC).astype(int)

print("기준 초과 측정:", df["over"].sum(), "회 / 화재 경보:", df["alarm"].sum(), "회")
if df["alarm"].any():
    first = df[df["alarm"] == 1].iloc[0]
    print(f"🔥 화재 경보 최초 발생: {first['time_s']:.1f}초 (불꽃 세기 {first['ir_pct']:.0f}%)")
print("햇빛 반사 구간(20~25초) 경보:", df[(df.time_s >= 20) & (df.time_s < 25)]["alarm"].sum(), "회 → 오경보 차단")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(df["time_s"], df["ir_pct"], "k-", lw=1.2, label="불꽃 세기")
ax[0].axhline(TH, color="gray", ls="--", label=f"기준 {TH}%")
ax[0].set_ylabel("%"); ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].set_title("KY-026 적외선 불꽃 감지")
ax[1].step(df["time_s"], df["alarm"], "k-", where="post")
ax[1].fill_between(df["time_s"], df["alarm"], step="post", color="lightgray")
ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(["정상", "경보"]); ax[1].set_xlabel("시간 (초)")
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "적외선IR센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "KY-026 불꽃 감지 모듈",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/flame_ir", payload)
# client.disconnect()

## 직접 해 보기
- `CONSEC`를 1로 바꾸면 햇빛 반사 구간에서 어떤 일이 생기나요?
- ADC 값을 10비트 정수(0~1023)로 바꿔 출력해 보세요. (`int(adc * 1023)`)
- 경보 중에는 GPIO18에 연결한 부저를 울리는 코드를 `gpiozero.Buzzer`로 작성해 보세요.